# 🏔️ Downloading DEMs from the OpenTopography Web API — Exercise Notebook

---

> **How to use this notebook:**  
> Each section contains **worked examples** you can run, followed by **practice exercises**.  
> Read the examples carefully, then attempt the exercises in the solution cells.  
> Use `Shift + Enter` to run a cell.

---

## 📋 Table of Contents

| # | Topic |
|---|-------|
| 0 | Setup & Environment |
| 1 | Core Imports |
| 2 | Function: `DownloadDemForState` — Download & Clip |
| 3 | Function: `DownloadDemByState` — Orchestrator |
| 4 | Function: `MergeDems` — Mosaic Multiple Tiles |
| 5 | Main Execution Block |

---

> **What this notebook does:**  
> Programmatically download a **Digital Elevation Model (DEM)** from the  
> [OpenTopography API](https://portal.opentopography.org/) and clip it  
> precisely to a state boundary polygon using **GeoPandas** and **Rasterio**.

---

# 0️⃣ Setup & Environment

---

> This notebook runs on **Google Colab**.  
> Data is read from and saved to **Google Drive** (`/content/drive/MyDrive/`).

| Library | Purpose |
|---------|----------|
| `geopandas` | Read state boundary shapefiles |
| `rasterio` | Read, clip, and write raster (DEM) files |
| `rasterio.mask` | Clip raster to vector polygon boundary |
| `rasterio.merge` | Merge / mosaic multiple DEM tiles |
| `requests` | Make HTTP calls to the OpenTopography API |
| `zipfile` | Unpack ZIP responses from the API |
| `io.BytesIO` | Handle binary download data in memory |
| `os` | Create folders and manage file paths |

> **API key required:**  
> Sign up at [portal.opentopography.org](https://portal.opentopography.org/) → Dashboard → copy your **API Key**.

---

### 💡 Example 0.1 — Install Rasterio

In [ ]:
pip install rasterio

### 💡 Example 0.2 — Mount Google Drive

In [ ]:
# Mount Drive before proceeding.
from google.colab import drive
drive.mount('/content/drive')

### 💡 Example 0.3 — Create Data & Output Folders

In [ ]:
import os

data_folder   = r'/content/drive/MyDrive/data'
output_folder = r'/content/drive/MyDrive/output'

for folder in [data_folder, output_folder]:
    if not os.path.exists(folder):
        os.mkdir(folder)
        print(f'Created: {folder}')
    else:
        print(f'Already exists: {folder}')

### ✏️ Exercise 0 — Practice Questions

**Q1.** After mounting Drive, run `!ls /content/drive/MyDrive/` to list your Drive contents. Do you see the `data` and `output` folders?

**Q2.** Create an additional sub-folder `output/dem_tiles` inside your output folder using `os.makedirs(..., exist_ok=True)`.

**Q3.** Confirm `rasterio` is installed by running `import rasterio; print(rasterio.__version__)`.

**Q4.** What does `exist_ok=True` do in `os.makedirs()`? Why is it safer than `os.mkdir()` when the folder may already exist?

**Q5.** Why do we mount Google Drive at the start of the notebook? What happens to files saved in `/content/` (not Drive) when the Colab session ends?


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import os, rasterio

# Q1:
# !ls /content/drive/MyDrive/

# Q2:
os.makedirs('/content/drive/MyDrive/output/dem_tiles', exist_ok=True)
print('dem_tiles folder ready')

# Q3:
print('Rasterio version:', rasterio.__version__)

# Q4:
# exist_ok=True → no error if the folder already exists.
# os.mkdir() raises FileExistsError if the folder is present.
# Use makedirs(exist_ok=True) for idempotent, re-runnable setup cells.

# Q5:
# Drive is mounted so files persist between sessions.
# /content/ is ephemeral — cleared when the Colab VM is recycled.
# Always save important outputs to /content/drive/MyDrive/.
```
</details>

---

# 1️⃣ Core Imports

---

> Import all libraries **once** at the top of the notebook so every subsequent cell can use them.

| Import | What it provides |
|--------|------------------|
| `geopandas as gpd` | `gpd.read_file()` to load shapefiles |
| `rasterio` | `rasterio.open()` to read/write GeoTIFFs |
| `rasterio.mask` | `.mask()` to clip raster to polygon |
| `rasterio.merge.merge` | `merge()` to mosaic DEM tiles |
| `requests` | `requests.get()` for HTTP API calls |
| `zipfile` | `zipfile.ZipFile()` to unpack `.zip` responses |
| `os` | `os.makedirs()`, `os.path.join()`, `os.remove()` |
| `io.BytesIO` | In-memory binary buffer for the download |

---

### 💡 Example 1.1 — Import All Required Libraries

In [ ]:
# Core Geospatial Libraries
import geopandas as gpd   # Read and handle vector data (shapefiles like state boundaries).
import rasterio            # Primary library for reading, writing, and manipulating raster data (DEMs).
import rasterio.mask       # Sub-module for clipping rasters to vector geometries.
from rasterio.merge import merge  # Sub-module for combining multiple rasters into a mosaic.

# System and Networking Libraries
import requests            # Make HTTP requests to download data from the OpenTopography API.
import zipfile             # Handle compressed files (some API downloads come as ZIP archives).
import os                  # Create folders and manage file paths.
from io import BytesIO     # Handle binary data in memory before saving to disk.

print('All libraries imported successfully ✅')
print(f'GeoPandas : {gpd.__version__}')
print(f'Rasterio  : {rasterio.__version__}')

### ✏️ Exercise 1 — Practice Questions

**Q1.** What is the purpose of `from io import BytesIO`? When do we need to handle binary data in memory instead of writing it directly to disk?

**Q2.** Why is `rasterio.mask` imported as a separate sub-module rather than being available from `import rasterio` alone?

**Q3.** What does `from rasterio.merge import merge` give us? Write a one-line description of what `merge()` does.

**Q4.** The `requests` library is used to call the OpenTopography API. What HTTP method does `requests.get()` use, and when would you use `requests.post()` instead?

**Q5.** Run `help(rasterio.mask.mask)` to read the docstring. List the three required arguments.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import rasterio.mask

# Q1:
# BytesIO wraps bytes in a file-like object so libraries (e.g. zipfile)
# can read from it without writing a temporary file to disk.
# Useful when the download is small or you want to avoid disk I/O.

# Q2:
# rasterio.mask is an optional sub-module not auto-imported with rasterio.
# Python only loads sub-modules when explicitly imported to keep memory lean.

# Q3:
# merge() mosaics a list of opened rasterio datasets into one array + transform.

# Q4:
# requests.get() → HTTP GET → safe, read-only, parameters in URL.
# requests.post() → HTTP POST → sends a request body; used for creating/submitting data.

# Q5:
help(rasterio.mask.mask)
# Required args: dataset (open rasterio file), shapes (geometries), crop (bool)
```
</details>

---

# 2️⃣ Function: `DownloadDemForState` — Download & Clip

---

> This is the **core function**. It:  
> 1. Builds the OpenTopography API URL from bounding box coordinates  
> 2. Sends an HTTP GET request and validates the response  
> 3. Handles both **GeoTIFF** and **ZIP** responses  
> 4. **Clips** the raw DEM to the exact state polygon boundary  
> 5. Saves the clipped DEM and cleans up temporary files  

| Step | What happens |
|------|-------------|
| Build URL | `SRTMGL1` DEM type + bounding box + `GTiff` format + API key |
| HTTP GET | `requests.get(Url)` → raises on non-200 status |
| Handle response | ZIP → extract `.tif`; else → write bytes directly |
| Clip | `rasterio.mask.mask(Src, [Geometry], crop=True)` |
| Update metadata | New `height`, `width`, `transform` from clipped extent |
| Save | `rasterio.open(OutputTif, 'w', **OutMeta)` |
| Cleanup | `os.remove(TempTif)` |

> **SRTM GL1** = Shuttle Radar Topography Mission, Global 1 arc-second (~30 m resolution)

---

### 💡 Example 2.1 — The Complete `DownloadDemForState` Function

In [ ]:
def DownloadDemForState(Geometry, Bounds, StateName, OutputFolder, ApiKey):
    """
    Downloads a raw DEM tile from OpenTopography and clips it to the state's
    boundary polygon.

    Args:
        Geometry (dict)    : GeoJSON dict of the state's boundary polygon.
        Bounds   (tuple)   : (minx, miny, maxx, maxy) bounding box.
        StateName (str)    : State name used for output file naming.
        OutputFolder (str) : Directory where files will be saved.
        ApiKey (str)       : Your personal OpenTopography API key.

    Returns:
        str: File path to the final clipped DEM.
    """
    # 1. Prepare Request Parameters
    MinX, MinY, MaxX, MaxY = Bounds

    # 2. Construct the OpenTopography API URL
    Url = (
        "https://portal.opentopography.org/API/globaldem"
        f"?demtype=SRTMGL1"                              # SRTM GL1 = 30 m resolution
        f"&south={MinY}&north={MaxY}&west={MinX}&east={MaxX}"  # Bounding box
        f"&outputFormat=GTiff"                           # Request GeoTIFF output
        f"&API_Key={ApiKey}"                             # Authentication
    )

    print(f"\nRequesting DEM from OpenTopography for: {StateName}")
    print(f"URL: {Url[:80]}...")

    # 3. Send the HTTP Request
    Response = requests.get(Url)
    Response.raise_for_status()   # Raises HTTPError for 4xx/5xx status codes

    # 4. Setup File Paths
    os.makedirs(OutputFolder, exist_ok=True)
    TempTif   = os.path.join(OutputFolder, f"{StateName}_raw.tif")
    OutputTif = os.path.join(OutputFolder, f"{StateName}_dem_clipped.tif")
    SourcePath = TempTif

    # 5. Handle ZIP or Direct GeoTIFF Response
    ContentType = Response.headers.get("Content-Type", "")
    if ContentType.startswith("application/zip") or Response.content[:2] == b"PK":
        print("  Response is a ZIP archive — extracting GeoTIFF...")
        with zipfile.ZipFile(BytesIO(Response.content)) as zf:
            TifNames = [n for n in zf.namelist() if n.endswith(".tif")]
            if not TifNames:
                raise ValueError(f"No .tif found inside ZIP for {StateName}")
            zf.extract(TifNames[0], OutputFolder)
            SourcePath = os.path.join(OutputFolder, TifNames[0])
    else:
        print("  Response is a direct GeoTIFF — saving...")
        with open(SourcePath, "wb") as f:
            f.write(Response.content)

    # 6. Clip DEM to the State Boundary
    print(f"  Clipping raw DEM to {StateName} boundary...")
    with rasterio.open(SourcePath) as Src:
        OutImage, OutTransform = rasterio.mask.mask(Src, [Geometry], crop=True)
        OutMeta = Src.meta.copy()

    # 7. Update Metadata
    OutMeta.update({
        "driver"   : "GTiff",
        "height"   : OutImage.shape[1],
        "width"    : OutImage.shape[2],
        "transform": OutTransform
    })

    # 8. Save the Clipped DEM
    with rasterio.open(OutputTif, "w", **OutMeta) as Dest:
        Dest.write(OutImage)
    print(f"  Clipped DEM saved to: {OutputTif}")

    # 9. Cleanup Temporary Raw File
    if SourcePath != OutputTif and os.path.exists(SourcePath):
        os.remove(SourcePath)
        print(f"  Temporary file removed: {SourcePath}")

    return OutputTif

### 💡 Example 2.2 — Anatomy of the OpenTopography API URL

In [ ]:
# Inspect the URL structure without making a real API call
example_bounds = (-104.05, 45.93, -96.55, 49.00)  # North Dakota bounding box
MinX, MinY, MaxX, MaxY = example_bounds
ApiKey = 'YOUR_API_KEY_HERE'

Url = (
    'https://portal.opentopography.org/API/globaldem'
    f'?demtype=SRTMGL1'
    f'&south={MinY}&north={MaxY}&west={MinX}&east={MaxX}'
    f'&outputFormat=GTiff'
    f'&API_Key={ApiKey}'
)

print('Constructed URL:')
print(Url)
print()
print('URL Parameters breakdown:')
print(f'  demtype      = SRTMGL1  (SRTM 30 m global)')
print(f'  south        = {MinY}   (min latitude)')
print(f'  north        = {MaxY}   (max latitude)')
print(f'  west         = {MinX}  (min longitude)')
print(f'  east         = {MaxX}   (max longitude)')
print(f'  outputFormat = GTiff    (GeoTIFF raster)')

### 💡 Example 2.3 — Understanding `rasterio.mask.mask()`

In [ ]:
# Conceptual walkthrough — no file needed
import rasterio.mask

print('rasterio.mask.mask() — key parameters:')
print()
print('  dataset  : an open rasterio DatasetReader (from rasterio.open())')
print('  shapes   : list of GeoJSON-like geometry dicts  ← [Geometry]')
print('  crop     : True → output is trimmed to geometry bounding box')
print('             False → output keeps original raster extent (masked with nodata)')
print()
print('Returns:')
print('  out_image     : NumPy array  (bands, rows, cols)')
print('  out_transform : Affine transform for the new clipped extent')
print()
print('Typical pattern:')
print("""
    with rasterio.open(path) as src:
        out_image, out_transform = rasterio.mask.mask(src, [geometry], crop=True)
        meta = src.meta.copy()
    meta.update({'height': out_image.shape[1],
                 'width' : out_image.shape[2],
                 'transform': out_transform})
    with rasterio.open(output_path, 'w', **meta) as dst:
        dst.write(out_image)
""")

### ✏️ Exercise 2 — Practice Questions

**Q1.** What does `Response.raise_for_status()` do? What exception does it raise, and when would you see it with the OpenTopography API?

**Q2.** The function checks `Response.content[:2] == b'PK'`. What is `b'PK'` and why does it identify a ZIP file?

**Q3.** Modify the URL construction to request **SRTMGL3** (90 m resolution) instead of SRTMGL1. What would the `demtype` parameter be?

**Q4.** What happens to the clipped raster's `height` and `width` compared to the raw downloaded DEM? Why must `OutMeta` be updated with the new values?

**Q5.** Why does the function call `os.remove(SourcePath)` at the end? What would happen if you skipped this cleanup step over many runs?


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
# Q1:
# raise_for_status() checks the HTTP status code.
# If status >= 400, it raises requests.exceptions.HTTPError.
# With OT API: 401 = invalid key, 400 = bad params, 429 = rate limit exceeded.

# Q2:
# b'PK' are the first two bytes (magic bytes) of every ZIP file.
# ZIP format always starts with the 'PK' signature (named after Phil Katz).
# This lets us detect ZIP content even if Content-Type header is wrong.

# Q3:
# Change demtype=SRTMGL1 → demtype=SRTMGL3
# SRTMGL3 = SRTM 3 arc-second (~90 m) — smaller file, lower resolution.

# Q4:
# Clipping changes the spatial extent → new pixel grid dimensions.
# Without updating height/width/transform in OutMeta,
# rasterio would try to write the wrong number of pixels → error or corrupt file.

# Q5:
# os.remove() deletes the raw unclipped temp file to save disk space.
# Skipping it would leave large raw DEM files accumulating on Drive
# (potentially gigabytes for large states or many runs).
```
</details>

---

# 3️⃣ Function: `DownloadDemByState` — Orchestrator

---

> This **orchestrator function** wraps `DownloadDemForState`.  
> It reads the shapefile, validates inputs, extracts the geometry and bounding box,  
> then calls the download function.

| Step | Code |
|------|------|
| Read shapefile | `gpd.read_file(ShapefilePath)` |
| Validate field | check `StateField in Gdf.columns` |
| Extract row | `Row = Gdf.iloc[0]` |
| Clean name | `str(Row[StateField]).replace(' ', '_')` |
| Get GeoJSON | `Row.geometry.__geo_interface__` |
| Get bounds | `Row.geometry.bounds` → `(minx, miny, maxx, maxy)` |
| Call download | `DownloadDemForState(Geometry, Bounds, ...)` |

---

### 💡 Example 3.1 — The Complete `DownloadDemByState` Function

In [ ]:
def DownloadDemByState(ShapefilePath, OutputFolder, ApiKey, StateField="NAME"):
    """
    Reads a shapefile representing a state boundary, extracts geometry and name,
    and initiates the DEM download and clipping process.

    Args:
        ShapefilePath (str) : Path to the state boundary shapefile (.shp).
        OutputFolder  (str) : Directory where the DEM will be saved.
        ApiKey        (str) : Your OpenTopography API key.
        StateField    (str) : Column name in the shapefile holding the state identifier.
                              Defaults to 'NAME'.

    Returns:
        str: File path to the final clipped DEM.
    """
    # 1. Read the Vector File
    Gdf = gpd.read_file(ShapefilePath)

    # 2. Input Validation
    if StateField not in Gdf.columns:
        raise ValueError(f"State field '{StateField}' not found in the shapefile. "
                         f"Available columns: {Gdf.columns.tolist()}")

    if len(Gdf) != 1:
        print(f"  Warning: Shapefile contains {len(Gdf)} features. Using the first row.")

    # 3. Extract State Data
    Row = Gdf.iloc[0]

    # Clean state name for safe file naming
    StateName = str(Row[StateField]).replace(" ", "_")

    # Convert Shapely geometry → GeoJSON dict (required by rasterio.mask)
    Geometry = Row.geometry.__geo_interface__

    # Get bounding box as (minx, miny, maxx, maxy)
    Bounds = Row.geometry.bounds

    print(f"State name   : {StateName}")
    print(f"Bounding box : {tuple(round(b, 4) for b in Bounds)}")
    print(f"CRS          : {Gdf.crs}")

    # 4. Initiate Download and Clip
    return DownloadDemForState(Geometry, Bounds, StateName, OutputFolder, ApiKey)

### 💡 Example 3.2 — Inspecting a Shapefile Before Calling the Function

In [ ]:
import geopandas as gpd

# Replace with your actual shapefile path
ShapefilePath = r'/content/drive/MyDrive/data/ND_Outline.shp'

try:
    Gdf = gpd.read_file(ShapefilePath)
    print('=== Shapefile Summary ===')
    print(f'Rows     : {len(Gdf)}')
    print(f'CRS      : {Gdf.crs}')
    print(f'Columns  : {Gdf.columns.tolist()}')
    print(f'Geom type: {set(Gdf.geom_type)}')
    print(f'Bounds   : {Gdf.total_bounds}')
    print()
    print('First row:')
    display(Gdf.head(1))
except Exception as e:
    print(f'Could not read shapefile: {e}')
    print('Make sure ND_Outline.shp is uploaded to your Drive data folder.')

### 💡 Example 3.3 — Understanding `__geo_interface__` and `.bounds`

In [ ]:
# Demonstrate what __geo_interface__ and .bounds return
from shapely.geometry import box

# Create a simple rectangular geometry for demonstration
sample_geom = box(-104.05, 45.93, -96.55, 49.00)  # North Dakota rough bbox

print('=== .bounds ===')
print(sample_geom.bounds)  # (minx, miny, maxx, maxy)
print(type(sample_geom.bounds))

print()
print('=== .__geo_interface__ (GeoJSON dict) ===')
geo = sample_geom.__geo_interface__
print(f'  type        : {geo["type"]}')
print(f'  coordinates : {str(geo["coordinates"])[:80]}...')
print()
print('This GeoJSON dict is what rasterio.mask.mask() expects as the [Geometry] list.')

### ✏️ Exercise 3 — Practice Questions

**Q1.** Open your ND shapefile and print all column names. Which column holds the state name — `'NAME'` or `'STATEFP'`? Why does this matter for `StateField`?

**Q2.** What does `.replace(' ', '_')` do to the state name, and why is this important when constructing a file path?

**Q3.** Print the output of `Row.geometry.__geo_interface__`. What is its `type` key? What format are the coordinates in?

**Q4.** What does `Row.geometry.bounds` return for North Dakota? Verify that the lat/lon values make geographic sense.

**Q5.** Modify `DownloadDemByState` to also print the **area** of the state polygon in km² (reproject to EPSG:5070 first). Add this as an extra print statement before calling `DownloadDemForState`.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import geopandas as gpd
from shapely.geometry import box

# Q1:
# gdf = gpd.read_file(ShapefilePath)
# print(gdf.columns.tolist())
# 'NAME' = human-readable ('North Dakota')
# 'STATEFP' = FIPS code ('38') — used when StateField='STATEFP'
# The field you pick becomes the output file name prefix.

# Q2:
# 'North Dakota'.replace(' ','_') → 'North_Dakota'
# Spaces in file names cause issues in command-line tools and some APIs.

# Q3:
sample = box(-104.05,45.93,-96.55,49.00)
geo = sample.__geo_interface__
print('type:', geo['type'])          # Polygon
print('coords type: list of rings') 
# Coordinates: list of (lon, lat) tuples — GeoJSON convention (lon first)

# Q4:
print(sample.bounds)  # (-104.05, 45.93, -96.55, 49.0)
# minx=-104.05 (western border), maxx=-96.55 (eastern border)
# miny=45.93  (southern border), maxy=49.0  (Canadian border) ✅

# Q5 — extra print in DownloadDemByState:
# gdf_proj = Gdf.to_crs(5070)
# area_km2 = gdf_proj.geometry.iloc[0].area / 1e6
# print(f'State area : {area_km2:,.0f} km²')
```
</details>

---

# 4️⃣ Function: `MergeDems` — Mosaic Multiple Tiles

---

> When a state is too large for a single API tile (or you download tiles separately),  
> `MergeDems` stitches all `.tif` files in a folder into one seamless raster.

| Step | Code |
|------|------|
| Find tiles | `[f for f in os.listdir(folder) if f.endswith('.tif')]` |
| Open all | `[rasterio.open(fp) for fp in DemFiles]` |
| Merge | `Mosaic, OutTrans = merge(SrcFiles)` |
| Copy meta | `SrcFiles[0].meta.copy()` |
| Update meta | new `height`, `width`, `transform` |
| Write output | `rasterio.open(OutputPath, 'w', **OutMeta)` |
| Close files | `for Src in SrcFiles: Src.close()` |

---

### 💡 Example 4.1 — The Complete `MergeDems` Function

In [ ]:
def MergeDems(InputFolder, OutputPath):
    """
    Merges all GeoTIFF DEM tiles found in a folder into a single mosaic raster.

    Args:
        InputFolder (str) : Path to the folder containing .tif DEM tiles.
        OutputPath  (str) : Full file path for the merged output GeoTIFF.

    Returns:
        None (saves the merged raster to OutputPath).
    """
    print(f"\nMerging DEMs from: {InputFolder}")

    # 1. Identify Input Files
    DemFiles = [
        os.path.join(InputFolder, f)
        for f in os.listdir(InputFolder)
        if f.endswith(".tif")
    ]
    if not DemFiles:
        raise FileNotFoundError(f"No .tif files found in: {InputFolder}")

    print(f"  Found {len(DemFiles)} tile(s): {[os.path.basename(f) for f in DemFiles]}")

    # 2. Open All Tile Files
    SrcFiles = [rasterio.open(fp) for fp in DemFiles]

    # 3. Perform Merge (Mosaic)
    # merge() returns: a NumPy array and the affine transform of the combined extent
    Mosaic, OutTrans = merge(SrcFiles)

    # 4. Prepare Output Metadata
    # Start from the first tile's metadata (CRS, dtype, nodata, etc.)
    OutMeta = SrcFiles[0].meta.copy()
    OutMeta.update({
        "driver"   : "GTiff",
        "height"   : Mosaic.shape[1],    # rows of the merged raster
        "width"    : Mosaic.shape[2],    # cols of the merged raster
        "transform": OutTrans            # updated georeference for the full extent
    })

    # 5. Save the Merged Raster
    with rasterio.open(OutputPath, "w", **OutMeta) as Dest:
        Dest.write(Mosaic)

    # 6. Close All Source Files (release file handles)
    for Src in SrcFiles:
        Src.close()

    print(f"  Merged DEM saved to: {OutputPath}")

### 💡 Example 4.2 — When to Use `MergeDems`

In [ ]:
# When would you use MergeDems?
# ─────────────────────────────
# Scenario A: State is too large for one API request.
#   → Split the bounding box into tiles (e.g. N/S halves), download each,
#     then merge into one seamless DEM.
#
# Scenario B: You already have multiple DEM tiles from different sources.
#   → Place all .tif files in one folder and call MergeDems().
#
# Scenario C: Multi-state analysis.
#   → Download one DEM per state, then merge all into a regional DEM.

print('MergeDems workflow:')
print('  1. Download tile_north.tif  → output/dem_tiles/')
print('  2. Download tile_south.tif  → output/dem_tiles/')
print('  3. MergeDems(')
print('       InputFolder = "output/dem_tiles",')
print('       OutputPath  = "output/nd_dem_merged.tif"')
print('     )')
print()
print('Result: one seamless GeoTIFF covering the entire state.')

### ✏️ Exercise 4 — Practice Questions

**Q1.** Why is it important to call `Src.close()` for every file in `SrcFiles` after merging? What problem can open file handles cause?

**Q2.** What happens if `InputFolder` contains `.tif` files with **different CRS values**? Will `merge()` handle this automatically or throw an error?

**Q3.** Modify `MergeDems` to also print the **final merged raster's dimensions** (rows × cols) and **CRS** after saving.

**Q4.** After merging, use `rasterio.open(OutputPath)` to verify the saved file. Print its `crs`, `res`, and the `min`/`max` of band 1.

**Q5.** Rewrite the file-finding list comprehension in step 1 to also include `.tiff` (with double `f`) files. Update the condition accordingly.


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import rasterio, numpy as np, os
from rasterio.merge import merge

# Q1:
# Unclosed file handles keep the file locked (especially on Windows).
# With many tiles, you can hit OS limits on open file descriptors.
# Always close files after merge() is complete.

# Q2:
# rasterio.merge() does NOT reproject automatically.
# If CRS values differ, the output may be incorrect or raise an error.
# Solution: reproject all tiles to a common CRS before merging.

# Q3 — extra prints in MergeDems after saving:
# with rasterio.open(OutputPath) as check:
#     print(f'  Merged CRS    : {check.crs}')
#     print(f'  Merged size   : {check.height} rows x {check.width} cols')

# Q4:
# with rasterio.open(output_path) as src:
#     data = src.read(1).astype(float)
#     print(src.crs, src.res)
#     print('min:', np.nanmin(data), 'max:', np.nanmax(data))

# Q5:
# DemFiles = [
#     os.path.join(InputFolder, f)
#     for f in os.listdir(InputFolder)
#     if f.endswith('.tif') or f.endswith('.tiff')
# ]
```
</details>

---

# 5️⃣ Main Execution Block

---

> This is the **only cell you need to edit** to run the full workflow.  
> Update the four student-specific inputs, then run this cell.

| Variable | What to set |
|----------|-------------|
| `ShapefileState` | Full path to your state `.shp` file on Google Drive |
| `OutputFolder` | Full path to the folder where the DEM will be saved |
| `ApiKey` | Your OpenTopography API key |
| `StateFieldAttribute` | Column name in the shapefile that holds the state ID |

> **How to get your API key:**  
> 1. Visit [portal.opentopography.org](https://portal.opentopography.org/)  
> 2. Create a free account → Dashboard → **API Key** tab  
> 3. Copy the key and paste it below

---

### 💡 Example 5.1 — The Main Execution Block

In [ ]:
if __name__ == "__main__":

    # ─────────────────────────────────────────────────
    # STUDENT-SPECIFIC INPUTS — Edit these four values
    # ─────────────────────────────────────────────────

    # 1. Path to your state boundary shapefile
    ShapefileState = r"/content/drive/MyDrive/data/ND_Outline.shp"

    # 2. Output folder where the clipped DEM will be saved
    OutputFolder = r"/content/drive/MyDrive/output"

    # 3. Your OpenTopography API key
    #    Get one at: https://portal.opentopography.org/ → Dashboard → API Key
    ApiKey = "YOUR_API_KEY_HERE"  # ← Replace with your actual key

    # 4. Column name in the shapefile that holds the state identifier
    #    Common values: 'NAME' (e.g. 'North Dakota') or 'STATEFP' (e.g. '38')
    StateFieldAttribute = "STATEFP"

    # ─────────────────────────────────────────────────

    print("--- Starting DEM Download and Clipping Process ---")
    print(f"  Shapefile  : {ShapefileState}")
    print(f"  Output     : {OutputFolder}")
    print(f"  State field: {StateFieldAttribute}")

    # Run the full workflow
    FinalDemPath = DownloadDemByState(
        ShapefileState,
        OutputFolder,
        ApiKey,
        StateField=StateFieldAttribute
    )

    print(f"\n✅ Process complete. Final DEM saved at:\n   {FinalDemPath}")

### 💡 Example 5.2 — Visualize the Downloaded DEM

In [ ]:
# Run this cell after the main block to visualize the clipped DEM
import rasterio
import numpy as np
import matplotlib.pyplot as plt

DemPath = FinalDemPath  # uses the path returned from the main block

with rasterio.open(DemPath) as src:
    print(f'CRS       : {src.crs}')
    print(f'Resolution: {src.res}')
    print(f'Size      : {src.width} x {src.height} pixels')
    print(f'Bands     : {src.count}')

    elev = src.read(1).astype(float)
    if src.nodata:
        elev[elev == src.nodata] = np.nan

print(f'\nElevation range: {np.nanmin(elev):.1f} – {np.nanmax(elev):.1f} m')
print(f'Mean elevation : {np.nanmean(elev):.1f} m')

fig, ax = plt.subplots(figsize=(10, 8))
img = ax.imshow(elev, cmap='terrain')
plt.colorbar(img, ax=ax, shrink=0.6, label='Elevation (m)')
ax.set_title(f'Downloaded DEM — {DemPath.split("/")[-1]}')
ax.set_axis_off()
plt.tight_layout()
plt.show()

### 💡 Example 5.3 — Verify File Was Saved to Drive

In [ ]:
import os

# Confirm the output file exists and print its size
if os.path.exists(FinalDemPath):
    size_mb = os.path.getsize(FinalDemPath) / (1024 * 1024)
    print(f'✅ File confirmed: {FinalDemPath}')
    print(f'   Size: {size_mb:.2f} MB')
else:
    print(f'❌ File not found: {FinalDemPath}')
    print('   Check that the main block ran successfully.')

# List all .tif files in the output folder
output_folder = os.path.dirname(FinalDemPath)
tif_files = [f for f in os.listdir(output_folder) if f.endswith('.tif')]
print(f'\nAll .tif files in {output_folder}:')
for f in tif_files:
    full = os.path.join(output_folder, f)
    print(f'  {f}  ({os.path.getsize(full)/1024:.0f} KB)')

### ✏️ Exercise 5 — Practice Questions

**Q1.** Run the main block with your own API key and `ND_Outline.shp`. Confirm the DEM file appears in your Drive output folder.

**Q2.** Change the `StateFieldAttribute` to `'NAME'` and re-run. How does the output filename change? Which field gives a more descriptive name?

**Q3.** After downloading, open the clipped DEM and compute the **histogram** of elevation values using `matplotlib`. How many pixels are at each 100 m interval?

**Q4.** Adapt the workflow to download a DEM for a **different US state** (e.g. Montana). What changes do you need to make — shapefile path, field name, output folder?

**Q5.** The `if __name__ == '__main__':` guard is used here. What does it do, and why is it good practice in Python scripts (even though it is not strictly required in a Jupyter notebook)?


In [ ]:
# 📝 Write your solutions here

# Q1:

# Q2:

# Q3:

# Q4:

# Q5:


<details>
<summary>🔍 Click to reveal solutions</summary>

```python
import rasterio, numpy as np, matplotlib.pyplot as plt, os

# Q2:
# With StateField='STATEFP': output file = '38_dem_clipped.tif'
# With StateField='NAME'   : output file = 'North_Dakota_dem_clipped.tif'
# NAME gives a more descriptive, human-readable filename.

# Q3:
# with rasterio.open(FinalDemPath) as src:
#     elev = src.read(1).astype(float)
#     if src.nodata: elev[elev==src.nodata]=np.nan
# valid = elev[~np.isnan(elev)].flatten()
# fig,ax = plt.subplots(figsize=(8,4))
# ax.hist(valid, bins=range(int(valid.min()),int(valid.max()),100),
#         color='steelblue', edgecolor='white')
# ax.set_xlabel('Elevation (m)'); ax.set_ylabel('Pixel count')
# ax.set_title('Elevation Distribution'); plt.show()

# Q4 — for Montana (state FIPS '30'):
# ShapefileState     = r'/content/drive/MyDrive/data/MT_Outline.shp'
# OutputFolder       = r'/content/drive/MyDrive/output'
# StateFieldAttribute = 'STATEFP'  # or 'NAME'
# (API key stays the same)

# Q5:
# if __name__ == '__main__': runs only when the file is executed directly,
# not when imported as a module.
# Good practice: prevents accidental execution when the file is imported.
# In Jupyter: the guard is always True, but it signals 'this is the entry point'.
```
</details>

---

# 📝 Chapter Review Questions

---

**1.** What does the OpenTopography `demtype=SRTMGL1` parameter mean? What resolution does it provide?

**2.** Why does the URL use `south`, `north`, `west`, `east` parameters instead of a polygon geometry?

**3.** Why is clipping (`rasterio.mask.mask`) necessary after downloading the bounding-box DEM?

**4.** What two things must you update in `OutMeta` after clipping, and why?

**5.** What is `BytesIO` used for, and when would you use it instead of writing to a temp file?

**6.** What is the difference between `rasterio.open(path)` (read) and `rasterio.open(path, 'w', **meta)` (write)?

**7.** Why must all rasterio source files be closed after `merge()` in `MergeDems`?

**8.** What change would you make to download a **90 m** DEM instead of a 30 m DEM from OpenTopography?

---

In [ ]:
# 📝 Write your answers here as comments

# 1:

# 2:

# 3:

# 4:

# 5:

# 6:

# 7:

# 8:


---

# 🎉 Congratulations — All 6 Sections Complete!

---

| ✅ | Section |
|----|----------|
| ✅ | 0 — Setup & Environment |
| ✅ | 1 — Core Imports |
| ✅ | 2 — `DownloadDemForState` — Download & Clip |
| ✅ | 3 — `DownloadDemByState` — Orchestrator |
| ✅ | 4 — `MergeDems` — Mosaic Multiple Tiles |
| ✅ | 5 — Main Execution Block |

---

## 🚀 Next Steps

| Topic | Libraries / APIs |
|-------|------------------|
| Visualize the DEM | `matplotlib`, `rasterio.plot` |
| Compute terrain derivatives | `numpy.gradient` (slope, aspect, hillshade) |
| Zonal stats over the DEM | `rasterstats` |
| Reproject to metric CRS | `rasterio.warp` (EPSG:5070) |
| Other DEM sources | `elevation` library, `NASADEM`, `Copernicus GLO-30` |
| Batch download (multi-state) | wrap `DownloadDemByState` in a loop |

---

### Happy Mapping! 🏔️💻

*Every pixel of elevation tells a story about the landscape beneath.*

---